# 🩸 Valoración del Riesgo de Anemia en Niños — Modelo de Machine Learning### Metodología CRISP-DM · Tesis UPAO 2026**Proyecto:** Aplicación móvil con técnicas de Machine Learning para la valoración del riesgo de anemia en niños menores de cinco años en Trujillo.**Autores:** Jara Tirado, Diego Francesco · Saldarriaga Urquizo, Pedro Leonardo**Asesor:** Cieza Mostacero, Segundo Edwin---Este notebook desarrolla las fases de CRISP-DM: Comprensión de los datos → Preparación → Modelado → Evaluación → Despliegue (serialización).Se entrenan y comparan **dos enfoques**:- **Modelo A (con hemoglobina):** máxima exactitud, pero la hemoglobina determina casi todo el diagnóstico.- **Modelo B (sin hemoglobina):** valoración de riesgo por *perfil* del niño (edad, CRED, altitud, etc.) — más realista.

## 0. Instalación e importación de librerías

In [ ]:
# Instalar dependencias (en Colab)!pip install -q imbalanced-learn xgboost shap

In [ ]:
import pandas as pdimport numpy as npimport matplotlib.pyplot as pltimport seaborn as snsimport warningswarnings.filterwarnings('ignore')from sklearn.model_selection import train_test_split, cross_val_scorefrom sklearn.preprocessing import StandardScalerfrom sklearn.linear_model import LogisticRegressionfrom sklearn.ensemble import RandomForestClassifierfrom sklearn.metrics import (classification_report, confusion_matrix,                             accuracy_score, roc_auc_score, ConfusionMatrixDisplay)from imblearn.over_sampling import SMOTEimport xgboost as xgbimport joblibsns.set(style='whitegrid')print('Librerías cargadas correctamente')

## 1. Carga del datasetSube el archivo **`trujillo_modelo.csv`** (dataset consolidado 2025 + 2026 de Trujillo, SIEN-HIS).En Colab: menú lateral → Archivos → subir, o usa el siguiente bloque.

In [ ]:
from google.colab import filesuploaded = files.upload()   # sube trujillo_modelo.csv

In [ ]:
df = pd.read_csv('trujillo_modelo.csv')print('Registros:', len(df))df.head()

## 2. Comprensión de los datos (CRISP-DM Fase 2)Exploración de las variables y la distribución de la variable objetivo.

In [ ]:
# Información generaldf.info()print('\nValores nulos por columna:')print(df.isnull().sum())

In [ ]:
# Distribución del diagnóstico originalprint(df['Dx_anemia'].value_counts())plt.figure(figsize=(7,4))df['Dx_anemia'].value_counts().plot(kind='bar', color=['#639922','#F5A623','#E8503A','#993333'])plt.title('Distribución del diagnóstico de anemia')plt.ylabel('Cantidad'); plt.xticks(rotation=20); plt.tight_layout(); plt.show()

In [ ]:
# Estadísticas descriptivasdf[['EdadMeses','Hemoglobina','AlturaREN','Cred','Suplementacion']].describe()

In [ ]:
# Matriz de correlaciónplt.figure(figsize=(6,5))num = df[['EdadMeses','Hemoglobina','AlturaREN','Cred','Suplementacion']].apply(pd.to_numeric, errors='coerce')sns.heatmap(num.corr(), annot=True, cmap='RdYlGn', center=0, fmt='.2f')plt.title('Matriz de correlación'); plt.tight_layout(); plt.show()

## 3. Preparación de los datos (CRISP-DM Fase 3)### 3.1 Reagrupación a 3 niveles de RIESGODado el fuerte desbalance y que la tesis es de *valoración de riesgo* (no diagnóstico), se reagrupan las 4 categorías en 3 niveles de riesgo.

In [ ]:
# Mapeo a niveles de riesgomapa = {'Normal':'Bajo', 'Anemia Leve':'Medio',        'Anemia Moderada':'Alto', 'Anemia Severa':'Alto'}df['Riesgo'] = df['Dx_anemia'].map(mapa)print(df['Riesgo'].value_counts())

### 3.2 Codificación y limpieza de variables

In [ ]:
# Convertir a numéricofor c in ['EdadMeses','Hemoglobina','AlturaREN','Cred','Suplementacion']:    df[c] = pd.to_numeric(df[c].astype(str).str.strip(), errors='coerce')# Codificar sexo (M=1, F=0)df['Sexo_num'] = (df['Sexo'].astype(str).str.strip()=='M').astype(int)# Eliminar filas sin datos clavedf = df.dropna(subset=['EdadMeses','AlturaREN','Cred','Suplementacion','Riesgo'])print('Registros tras limpieza:', len(df))

### 3.3 Definición de los dos conjuntos de variables- **Enfoque A (con hemoglobina):** incluye la hemoglobina → muy preciso pero la hemoglobina determina el diagnóstico.- **Enfoque B (sin hemoglobina):** solo factores de perfil → valoración de riesgo real.

In [ ]:
feats_A = ['EdadMeses','Hemoglobina','AlturaREN','Sexo_num','Cred','Suplementacion']feats_B = ['EdadMeses','AlturaREN','Sexo_num','Cred','Suplementacion']  # sin hemoglobinay = df['Riesgo']

### 3.4 División train/test (70-30) y balanceo con SMOTE**Importante:** SMOTE se aplica SOLO al conjunto de entrenamiento, nunca al de prueba, para no sesgar la evaluación.

In [ ]:
def preparar(feats):    X = df[feats]    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.30,                                          stratify=y, random_state=42)    # SMOTE solo en train    sm = SMOTE(random_state=42, k_neighbors=5)    Xtr_bal, ytr_bal = sm.fit_resample(Xtr, ytr)    return Xtr_bal, Xte, ytr_bal, yteprint('Enfoque A (con Hb):')XtrA, XteA, ytrA, yteA = preparar(feats_A)print('  Train balanceado:', pd.Series(ytrA).value_counts().to_dict())print('\nEnfoque B (sin Hb):')XtrB, XteB, ytrB, yteB = preparar(feats_B)print('  Train balanceado:', pd.Series(ytrB).value_counts().to_dict())

## 4. Modelado (CRISP-DM Fase 4)Se entrenan tres algoritmos: Regresión Logística (línea base), Random Forest y XGBoost.

In [ ]:
def entrenar_modelos(Xtr, ytr):    # Codificar etiquetas para XGBoost    modelos = {        'Regresión Logística': LogisticRegression(max_iter=1000, class_weight='balanced'),        'Random Forest': RandomForestClassifier(n_estimators=150, max_depth=12,                                                class_weight='balanced', random_state=42),    }    entrenados = {}    for nombre, m in modelos.items():        m.fit(Xtr, ytr)        entrenados[nombre] = m    # XGBoost (requiere etiquetas numéricas)    clases = sorted(ytr.unique())    mapa_c = {c:i for i,c in enumerate(clases)}    xgbm = xgb.XGBClassifier(n_estimators=150, max_depth=6, learning_rate=0.1,                             random_state=42, eval_metric='mlogloss')    xgbm.fit(Xtr, ytr.map(mapa_c))    entrenados['XGBoost'] = (xgbm, mapa_c)    return entrenados

In [ ]:
print('Entrenando modelos del Enfoque B (sin hemoglobina)...')modelos_B = entrenar_modelos(XtrB, ytrB)print('Entrenando modelos del Enfoque A (con hemoglobina)...')modelos_A = entrenar_modelos(XtrA, ytrA)print('Modelos entrenados.')

## 5. Evaluación (CRISP-DM Fase 5)Se evalúan los modelos en el conjunto de prueba con métricas de clasificación y matriz de confusión.

In [ ]:
def evaluar(modelo, Xte, yte, nombre):    if isinstance(modelo, tuple):  # XGBoost        xgbm, mapa_c = modelo        inv = {v:k for k,v in mapa_c.items()}        pred = pd.Series(xgbm.predict(Xte)).map(inv).values    else:        pred = modelo.predict(Xte)    acc = accuracy_score(yte, pred)    print(f'\n=== {nombre} ===')    print(f'Accuracy: {acc:.3f}')    print(classification_report(yte, pred, zero_division=0))    return acc, pred

In [ ]:
print('############ ENFOQUE B — SIN HEMOGLOBINA (valoración de riesgo real) ############')resB = {}for nombre, m in modelos_B.items():    acc, pred = evaluar(m, XteB, yteB, nombre)    resB[nombre] = acc

In [ ]:
print('############ ENFOQUE A — CON HEMOGLOBINA (referencia) ############')resA = {}for nombre, m in modelos_A.items():    acc, pred = evaluar(m, XteA, yteA, nombre)    resA[nombre] = acc

### 5.1 Comparación de los modelos

In [ ]:
comp = pd.DataFrame({'Sin Hemoglobina (Enfoque B)': resB,                     'Con Hemoglobina (Enfoque A)': resA})print(comp.round(3))comp.plot(kind='bar', figsize=(8,4), color=['#2E75B6','#639922'])plt.title('Comparación de accuracy por modelo y enfoque')plt.ylabel('Accuracy'); plt.xticks(rotation=15); plt.ylim(0,1.05)plt.legend(); plt.tight_layout(); plt.show()

> **Nota de interpretación:** el Enfoque A alcanza una exactitud muy alta porque la hemoglobina determina directamente el diagnóstico (riesgo de *data leakage*). El Enfoque B, sin hemoglobina, refleja la verdadera capacidad del modelo para valorar el riesgo a partir del perfil del niño, y es el más apropiado para los fines de la investigación.

### 5.2 Matriz de confusión del modelo seleccionado (Random Forest — Enfoque B)

In [ ]:
rf_B = modelos_B['Random Forest']pred_B = rf_B.predict(XteB)cm = confusion_matrix(yteB, pred_B, labels=['Bajo','Medio','Alto'])disp = ConfusionMatrixDisplay(cm, display_labels=['Bajo','Medio','Alto'])disp.plot(cmap='Blues'); plt.title('Matriz de confusión — Random Forest (sin Hb)')plt.tight_layout(); plt.show()

## 6. Interpretabilidad — ¿Qué factores influyen en el riesgo?Esta es la pregunta clave: además de predecir, el modelo explica **qué factores** pesan más.

### 6.1 Importancia global de variables (Feature Importance)

In [ ]:
importancias = pd.Series(rf_B.feature_importances_, index=feats_B).sort_values(ascending=True)plt.figure(figsize=(7,4))importancias.plot(kind='barh', color='#0F6E56')plt.title('Factores que más influyen en el riesgo de anemia (sin Hb)')plt.xlabel('Importancia'); plt.tight_layout(); plt.show()print(importancias.sort_values(ascending=False))

### 6.2 Explicación por caso individual con SHAPSHAP explica, para un niño concreto, por qué el modelo le asignó ese nivel de riesgo.

In [ ]:
import shapexplainer = shap.TreeExplainer(rf_B)shap_values = explainer.shap_values(XteB.iloc[:200])# Resumen globalshap.summary_plot(shap_values, XteB.iloc[:200], feature_names=feats_B, show=True)

## 7. Despliegue (CRISP-DM Fase 6) — Serialización del modeloSe guarda el modelo final para integrarlo en el backend de la aplicación.

In [ ]:
# Guardar el modelo seleccionado (Random Forest, enfoque B)joblib.dump(rf_B, 'modelo_anemia_rf.pkl')print('Modelo guardado como modelo_anemia_rf.pkl')# Descargar en Colabfrom google.colab import filesfiles.download('modelo_anemia_rf.pkl')

In [ ]:
# Ejemplo de uso del modelo guardadomodelo = joblib.load('modelo_anemia_rf.pkl')# Niño ejemplo: edad 36 meses, altitud 34, sexo M, sin CRED, con suplementaciónejemplo = pd.DataFrame([[36, 34, 1, 0, 1]], columns=feats_B)riesgo = modelo.predict(ejemplo)[0]proba = modelo.predict_proba(ejemplo).max()print(f'Riesgo predicho: {riesgo}  (probabilidad {proba:.2f})')

## 8. Conclusiones- Se consolidó un dataset de **55,207 registros** de Trujillo (SIEN-HIS 2025 + 2026).- La variable objetivo se reagrupó en **3 niveles de riesgo** (Bajo, Medio, Alto).- El fuerte desbalance se trató con **SMOTE** (solo en entrenamiento).- Se compararon **dos enfoques**: con y sin hemoglobina, evidenciando el riesgo de *data leakage*.- El modelo de **valoración de riesgo** (sin hemoglobina) identifica como factores más influyentes la **edad**, el **control CRED** y la **altitud**.- El modelo se serializó (`.pkl`) para su integración en la aplicación móvil.---*Notebook desarrollado siguiendo la metodología CRISP-DM.*